# Large Run Step 3: Metric Task Planning and Slurm Execution

Purpose: plan metric tasks from preprocessed waveform metadata, submit metric batches to Slurm, merge completed batches, and write downstream metric outputs without running heavy calculations in the notebook.

Outputs: metric waveform inventories, task manifest, Slurm array script, merged metric rows, `metrics_long.parquet`, `path_table.parquet`, dashboard datasets, and summaries.


## Setup
Purpose: load config/output paths and shared settings.

Outputs: printed paths and helper variables.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import time

import pandas as pd

from spatial_vtk.config import SpatialVTKConfig
from spatial_vtk.config.outputs import resolve_output_path
from spatial_vtk.io import load_output_table, preview_output_table, write_output_table


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "spatial_vtk").exists():
            return candidate
    return start


repo_root = find_repo_root()
config_candidates = [
    repo_root / "runs" / "spatial_vtk_config.yaml",
    Path.cwd() / "spatial_vtk_config.yaml",
    Path.cwd() / "runs" / "spatial_vtk_config.yaml",
]
config_path = next((path for path in config_candidates if path.exists()), config_candidates[0])
cfg = SpatialVTKConfig.from_file(config_path).activate()

outputs_root = Path(cfg.path("outputs.root", create_parent=True) or (cfg.root_dir / "runs" / "outputs"))
tables_dir = Path(cfg.path("outputs.tables", create_parent=True) or (outputs_root / "tables"))
figures_dir = Path(cfg.path("outputs.figures", create_parent=True) or (outputs_root / "figures"))
slurm_dir = outputs_root / "slurm"
logs_dir = outputs_root / "logs"
for directory in (tables_dir, figures_dir, slurm_dir, logs_dir):
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOCAL = os.environ.get("SVTK_RUN_LOCAL", "0") == "1"
SUBMIT_SLURM = os.environ.get("SVTK_SUBMIT_SLURM", "0") == "1"
OVERWRITE = os.environ.get("SVTK_OVERWRITE", "0") == "1"
QC_CHUNKSIZE = int(os.environ.get("SVTK_QC_CHUNKSIZE", "1000000"))
PREVIEW_ROWS = int(os.environ.get("SVTK_PREVIEW_ROWS", "5"))

print(f"repo_root: {repo_root}")
print(f"config_path: {config_path}")
print(f"outputs_root: {outputs_root}")
print(f"tables_dir: {tables_dir}")
print(f"figures_dir: {figures_dir}")
print(f"SUBMIT_SLURM={SUBMIT_SLURM} RUN_LOCAL={RUN_LOCAL} OVERWRITE={OVERWRITE}")


## Helpers
Purpose: define skip/status/Slurm helpers for this notebook.

Outputs: helper functions only.


In [ ]:
def exists(path):
    return Path(path).exists()


def should_run(*paths):
    return OVERWRITE or not all(Path(path).exists() for path in paths)


def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"path": str(path), "exists": False, "size_gb": None, "modified": None}
    return {
        "path": str(path),
        "exists": True,
        "size_gb": round(path.stat().st_size / 1024**3, 3),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }


def show_files(paths):
    display(pd.DataFrame([file_info(path) for path in paths]))


def submit_script(script_path, *, submit=SUBMIT_SLURM):
    script_path = Path(script_path)
    print(f"script: {script_path}")
    if submit:
        result = subprocess.run(["sbatch", str(script_path)], text=True, capture_output=True, check=False)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"sbatch failed with return code {result.returncode}")
    else:
        print(f"Set SVTK_SUBMIT_SLURM=1 and rerun this cell, or submit manually:")
        print(f"sbatch {shlex.quote(str(script_path))}")


def write_python_slurm_script(script_name, python_body, *, job_name, walltime="24:00:00", memory="32G", cpus=1):
    script_path = slurm_dir / script_name
    body = textwrap.dedent(python_body).strip()
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={logs_dir}/{job_name}_%j.out
#SBATCH --error={logs_dir}/{job_name}_%j.err
#SBATCH --time={walltime}
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}

set -euo pipefail
cd {repo_root}
source /project2/jvidale_1700/miniforge3/etc/profile.d/conda.sh || true
conda activate spatial-vtk-py312 || true
python - <<'PYJOB'
{body}
PYJOB
"""
    script_path.write_text(script, encoding="utf-8")
    script_path.chmod(0o755)
    return script_path


def preview_table_path(path, n=PREVIEW_ROWS, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq
        parquet = pq.ParquetFile(path)
        available = parquet.schema.names
        selected = [column for column in (columns or available) if column in available]
        for batch in parquet.iter_batches(batch_size=max(int(n), 1), columns=selected):
            return batch.to_pandas().head(n)
        return pd.DataFrame(columns=selected)
    return pd.read_csv(path, nrows=n, usecols=columns)


## Resolve Metric Workflow Outputs
Purpose: identify required metric workflow files and show current completion status.

Outputs: status table only.


In [ ]:
trace_metadata_path = outputs_root / "preprocessed_waveforms" / "metadata" / "trace_metadata_preprocessed.csv"
observed_inventory_path = tables_dir / "observed_metric_inventory.parquet"
synthetic_inventory_path = tables_dir / "synthetic_metric_inventory.parquet"
qc_inventory_overlap_path = resolve_output_path("qc_inventory_overlap", kind="table", create_parent=True)
metric_manifest_path = tables_dir / "metric_manifest.json"
metric_batches_dir = outputs_root / "metric_batches"
metric_rows_path = tables_dir / "metric_rows.parquet"
metrics_long_path = resolve_output_path("metrics_long", kind="table", create_parent=True)
metrics_enriched_path = resolve_output_path("metrics_enriched", kind="table", create_parent=True)
path_table_path = resolve_output_path("path_table", kind="table", create_parent=True)
path_summary_path = resolve_output_path("path_summary", kind="table", create_parent=True)

show_files([
    trace_metadata_path,
    observed_inventory_path,
    synthetic_inventory_path,
    qc_inventory_overlap_path,
    metric_manifest_path,
    metric_rows_path,
    metrics_long_path,
    metrics_enriched_path,
    path_table_path,
    path_summary_path,
])


## Build Metric Waveform Inventories
Purpose: derive observed and synthetic metric inventories from preprocessed trace metadata. This is submitted when inventories are missing.

Outputs: `observed_metric_inventory.parquet` and `synthetic_metric_inventory.parquet`.


In [ ]:
if not trace_metadata_path.exists():
    print(f"Trace metadata is not ready yet: {trace_metadata_path}")
elif should_run(observed_inventory_path, synthetic_inventory_path):
    script = write_python_slurm_script(
        "step03_metric_inventories.slurm",
        f"""
        from pathlib import Path
        import pandas as pd
        from spatial_vtk.config import SpatialVTKConfig

        cfg = SpatialVTKConfig.from_file({str(config_path)!r}).activate()
        trace_metadata_path = Path({str(trace_metadata_path)!r})
        observed_inventory_path = Path({str(observed_inventory_path)!r})
        synthetic_inventory_path = Path({str(synthetic_inventory_path)!r})
        columns = ["source_type", "event_id", "station", "component", "output_file", "delta", "sampling_rate", "starttime", "endtime"]
        metadata = pd.read_csv(trace_metadata_path, usecols=lambda column: column in columns, low_memory=False)
        metadata = metadata.rename(columns={{"source_type": "source", "output_file": "waveform_path", "delta": "dt"}})
        models = cfg.section("metrics.models", []) or []
        synthetic_model = str(models[0]) if len(models) == 1 else ""
        metadata["model"] = ""
        metadata.loc[metadata["source"].astype(str).str.lower().eq("synthetic"), "model"] = synthetic_model
        keep = ["source", "event_id", "station", "component", "model", "waveform_path", "dt", "sampling_rate", "starttime", "endtime"]
        metadata = metadata.loc[:, [column for column in keep if column in metadata.columns]].drop_duplicates()
        observed = metadata.loc[metadata["source"].astype(str).str.lower().eq("observed")].copy()
        synthetic = metadata.loc[metadata["source"].astype(str).str.lower().eq("synthetic")].copy()
        observed_inventory_path.parent.mkdir(parents=True, exist_ok=True)
        observed.to_parquet(observed_inventory_path, index=False)
        synthetic.to_parquet(synthetic_inventory_path, index=False)
        print(f"observed rows={{len(observed)}} synthetic rows={{len(synthetic)}}")
        """,
        job_name="svtk-step03-inventory",
        walltime="04:00:00",
        memory="32G",
        cpus=1,
    )
    submit_script(script)
else:
    print("Metric waveform inventories already exist; skipping.")


## Plan Metric Tasks
Purpose: create a manifest from observed/synthetic inventories and the overlap QC sidecar. Planning is local because it only writes a manifest and does not load waveform files.

Outputs: `metric_manifest.json` and batch output paths under `metric_batches/`.


In [ ]:
if not all(path.exists() for path in [observed_inventory_path, synthetic_inventory_path, qc_inventory_overlap_path]):
    print("Metric inventories or overlap QC are not ready yet.")
elif should_run(metric_manifest_path):
    cmd = [
        "svtk", "metrics", "plan",
        "--config", str(config_path),
        "--observed-inventory", str(observed_inventory_path),
        "--synthetic-inventory", str(synthetic_inventory_path),
        "--qc-table", str(qc_inventory_overlap_path),
        "--manifest",
        "--batch-output-dir", str(metric_batches_dir),
        "--batch-size", os.environ.get("SVTK_METRIC_BATCH_SIZE", "100"),
        "--output", str(metric_manifest_path),
    ]
    print(" ".join(shlex.quote(part) for part in cmd))
    if RUN_LOCAL:
        subprocess.run(cmd, check=True)
    else:
        print("Set SVTK_RUN_LOCAL=1 to run planning from this notebook, or run the printed command on the login node.")
else:
    print("Metric manifest already exists; skipping planning.")


## Submit Metric Slurm Array
Purpose: write and optionally submit the metric batch array. Each array task runs one manifest batch and skips completed batch outputs unless overwritten.

Outputs: Slurm array script and per-batch metric outputs.


In [ ]:
if not metric_manifest_path.exists():
    print("Metric manifest is not ready yet.")
else:
    from spatial_vtk.metrics.workflow.slurm import slurm_settings_from_config, write_metrics_slurm_script
    settings = slurm_settings_from_config(cfg)
    script_path = slurm_dir / "step03_run_metrics.slurm"
    write_metrics_slurm_script(metric_manifest_path, script_path, settings)
    submit_script(script_path)


## Merge Metric Batches
Purpose: merge completed metric batch outputs into one metric-row table. This should be run after the Slurm array finishes.

Outputs: `metric_rows.parquet`.


In [ ]:
if not metric_manifest_path.exists():
    print("Metric manifest is not ready yet.")
elif should_run(metric_rows_path):
    cmd = ["svtk", "metrics", "merge-batches", "--manifest", str(metric_manifest_path), "--output", str(metric_rows_path)]
    print(" ".join(shlex.quote(part) for part in cmd))
    if RUN_LOCAL:
        subprocess.run(cmd, check=True)
    else:
        print("Set SVTK_RUN_LOCAL=1 after Slurm batches finish, or run the printed command on the login node.")
else:
    print("Merged metric rows already exist; skipping.")


## Write Downstream Metric Outputs
Purpose: convert merged metric rows into standard tables and dashboard datasets in a separate command.

Outputs: `metrics_long.parquet`, `path_table.parquet`, `path_summary.parquet`, dashboard metrics, and dashboard summaries.


In [ ]:
if not metric_rows_path.exists():
    print("Merged metric rows are not ready yet.")
elif should_run(metrics_long_path, path_table_path, path_summary_path):
    cmd = [
        "svtk", "metrics", "outputs",
        "--metrics", str(metric_rows_path),
        "--events", str(resolve_output_path("prepared_events", kind="table", create_parent=True)),
        "--stations", str(resolve_output_path("prepared_stations", kind="table", create_parent=True)),
        "--output-dir", str(tables_dir),
        "--format", "parquet",
        "--dashboard-partitioned",
    ]
    print(" ".join(shlex.quote(part) for part in cmd))
    if RUN_LOCAL:
        subprocess.run(cmd, check=True)
    else:
        print("Set SVTK_RUN_LOCAL=1 to write downstream outputs, or run the printed command on the login node.")
else:
    print("Downstream metric outputs already exist; skipping.")


## Preview Metric Outputs
Purpose: inspect only a small row sample from the long metric table.

Outputs: bounded table preview.


In [ ]:
if metrics_long_path.exists():
    display(preview_table_path(metrics_long_path, PREVIEW_ROWS))
else:
    print(f"Metric output is not ready yet: {metrics_long_path}")
